In [31]:
import pandas as pd
import string
import ast
import os
from itertools import chain

from tqdm import tqdm

from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader
import transformers

## Data Cleaning

In [32]:
# Using os.path - go up one directory and access the file
data_path = os.path.join(os.path.dirname(os.path.abspath('')), 'job_descriptions.csv')
df = pd.read_csv(data_path)

In [33]:
# drop the url since it is not a relevant predictor
clean_df = df.drop('url', axis=1)

# split data into jobs and skills
jobs = clean_df['job']                  
skills = clean_df['skills']

### Job Description

We tokenize by word and add a `job_id` column.

In [34]:
# Standardize punctuation to prevent random biases.

punkt = string.punctuation
punkt = punkt[0:2] + punkt[3:13] + punkt[14:-1]

for i in range(len(jobs)):
    jobs.loc[i] = jobs.loc[i].lower()

    punkt_str = jobs.loc[i]
    test_str = punkt_str.translate(str.maketrans('', '', punkt))
    jobs.loc[i] = test_str

In [35]:
for i in range(len(jobs)):
    jobs.loc[i] = jobs[i].split()

jobs = jobs.explode(ignore_index=False).rename_axis('job_id').reset_index()

In [36]:
jobs[jobs['job_id'] == 0]

,job_id,job
0,0,summarythe
1,0,database
2,0,developer
3,0,is
4,0,part
...,...,...
713,0,after
714,0,a
715,0,project
716,0,is


### Skills

The skills data is in a `string` format, thus we convert from a string to a list using this claude generated function.

In [37]:
# Standardize punctuation to prevent random biases.
punkt_skills = punkt[:10] + punkt[11:]

for i in range(len(skills)):
    skills.loc[i] = skills.loc[i].lower()

    punkt_str = skills.loc[i]
    test_str = punkt_str.translate(str.maketrans('', '', punkt_skills))
    skills.loc[i] = test_str

In [38]:
# Assuming df is your dataframe and 'skills' is your column name
def convert_string_to_list(skills_str):
    try:
        # For properly formatted strings like "[item1, item2, item3]"
        return ast.literal_eval(skills_str)
    except (ValueError, SyntaxError):
        # Handle potential errors or edge cases
        if isinstance(skills_str, list):
            # If it's already a list, return it as is
            return skills_str
        elif isinstance(skills_str, str):
            # For simple comma-separated strings without brackets
            if skills_str.strip() == '':
                return []
            if '[' not in skills_str and ']' not in skills_str:
                return [item.strip() for item in skills_str.split(',')]
        # Default fallback
        return []

# Apply the conversion to the skills column
skills = skills.apply(convert_string_to_list)

### Adding features

We conduct IOB tagging to add further features into our data increasing the performance of our model.

In [39]:
def split_and_tag(skills_list):
    """
    Split a skills list into single words, then give them the IOB tags.
    
    """

    split_words = []
    tags = []
    
    for skill in skills_list:
        words = skill.split()
        split_words.extend(words)
        
        if len(words) == 1:
            tags.append(2)
        else:
            # First word gets 'B', subsequent words get 'I'
            tags.append(2)
            tags.extend([0] * (len(words) - 1))
    
    skill_tag_dict = {
        'skills' : split_words,
        'tags' : tags
    }

    return skill_tag_dict

In [40]:
def get_tag(curr_dict: dict, search_str: str):
    tagged_list = []
    for word in search_str:
        if word in curr_dict['skills']:
            idx = curr_dict['skills'].index(word)
            tag = curr_dict['tags'][idx]
            tagged_list.append(tag)
        else:
            tagged_list.append(1)
        
    return tagged_list

In [41]:
big_list = []
for i in range(len(skills)):
    in_skill  = skills[i]
    curr_dict = split_and_tag(in_skill)
    jd = jobs[jobs['job_id'] == i]['job']
    tagged_list = get_tag(curr_dict, jd)
    big_list.append(tagged_list)

unpacked_list = list(chain.from_iterable(big_list))

In [42]:
jobs['tags'] = pd.Series(unpacked_list, index=jobs.index)

In [43]:
data_gr = jobs.groupby("job_id").agg({'job': list, 'tags':list})

In [44]:
data_gr

,job,tags
job_id,,
0,"[summarythe, database, developer, is, part, of...","[1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, ..."
1,"[come, work, with, usat, rbc, our, culture, is...","[1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
2,"[job, introductionyou’ll, be, in, an, exciting...","[1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,"[the, information, technology, it, department,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,"[are, you, an, analytics, professional, experi...","[1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, ..."
...,...,...
1837,"[in, this, role, you, will, focus, on, develop...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 0, 1, 1, 1, ..."
1838,"[trident, consulting, is, seeking, a, machine,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1839,"[were, looking, for, a, passionate, software, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."


In [45]:
train_sent, val_sent, train_tag, val_tag = train_test_split(data_gr['job'], data_gr['tags'], test_size=0.01, random_state=10)